# Time Travel Contamination Probe (v3)

Golchin & Surdeanu (2024), *Time Travel in LLMs*: give a model a truncated passage, once with the source
named ("guided") and once without ("general"), and compare how closely each completion matches the real
continuation. A gap between guided and general overlap on the **original** puzzle, that collapses on the
**adapted** puzzle, is evidence the model recognizes the original text specifically rather than just
generating plausible GRE-style prose.

**What's different in this version (v3):**
- Every model's results are kept in one running cache file, `time_travel_results_master.csv`. Before
  running any model -- API or Ollama -- the notebook checks whether that `(backend, model)` already has
  rows in the cache and skips it if so. Re-running this notebook top to bottom after a partial run (or
  after adding a new model to the list) only runs what's missing.
- Your existing gpt-4o/o3 run (`time_travel_results_20261010_005717.csv`) seeds the cache automatically
  the first time this notebook runs, so those two never get re-run unless you delete the cache file.
- Anthropic and Google clients are no longer silently swallowed on import failure (the v1 bug): packages
  are installed up front and imported directly, so a missing dependency raises immediately instead of
  quietly turning into `ERROR:` completions and NaN scores three cells later.
- Ollama models still pull -> run -> remove, one at a time -- but now skip the pull entirely if that
  model's results are already cached.

**Three puzzle groups, scored differently:**
- `success` (10) -- original + a judge/human-confirmed adaptation. Full guided-vs-general comparison on both.
- `partial_fail` (1) -- original + an adaptation that parsed but didn't pass judging.
- `empty_fail` (10) -- original only; no parseable adaptation, so no adapted-side comparison for these.

**Before running:** put `contamination_dataset.csv` and `time_travel_results_20261010_005717.csv` in the
same folder as this notebook, and fill in your API keys in the Setup section below.

## 0. Setup — install everything this notebook needs

In [ ]:
# Python packages
%pip install -q openai anthropic google-genai rouge-score tenacity tqdm pandas

# Ollama CLI (skip if already installed -- the install script is safe to re-run)
!curl -fsSL https://ollama.com/install.sh | sh

!which ollama || echo "ollama still not on PATH -- installation above did not succeed, fix before running the Ollama section"

In [ ]:
# Start the Ollama server in the background. If you already have `ollama serve` running
# elsewhere (e.g. a separate terminal), skip this cell -- it detects that and won't start a second one.
import subprocess, time, requests

def ollama_is_up():
    try:
        return requests.get("http://localhost:11434/api/tags", timeout=2).ok
    except Exception:
        return False

if ollama_is_up():
    print("Ollama server already running.")
    ollama_proc = None
else:
    ollama_proc = subprocess.Popen(["ollama", "serve"],
                                    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    for _ in range(15):
        time.sleep(1)
        if ollama_is_up():
            print("Ollama server is up.")
            break
    else:
        print("Ollama server did not come up after 15s -- check manually before running the Ollama section.")

## 1. API keys and clients

In [ ]:
import os
from datetime import datetime

OPENAI_API_KEY    = os.environ.get("OPENAI_API_KEY", "")
ANTHROPIC_API_KEY = os.environ.get("ANTHROPIC_API_KEY", "")
GOOGLE_API_KEY    = os.environ.get("GOOGLE_API_KEY", "")
OLLAMA_URL        = "http://localhost:11434/v1"
RUN_ID            = datetime.now().strftime("%Y%m%d_%H%M%S")

# If os.environ isn't picking these up (e.g. keys were set after this kernel started, or
# added as Colab secrets rather than env vars), paste them directly here instead:
# OPENAI_API_KEY    = "sk-..."
# ANTHROPIC_API_KEY = "sk-ant-..."
# GOOGLE_API_KEY    = "AIza..."

from openai import OpenAI as _OpenAIClient
openai_client  = _OpenAIClient(api_key=OPENAI_API_KEY)
ollama_client  = _OpenAIClient(base_url=OLLAMA_URL, api_key="local")

import anthropic
anthropic_client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

from google import genai
google_client = genai.Client(api_key=GOOGLE_API_KEY)

# Sanity check BEFORE burning any API calls -- confirms keys are actually loaded and
# clients actually constructed.
print("OpenAI key set:    ", bool(OPENAI_API_KEY))
print("Anthropic key set: ", bool(ANTHROPIC_API_KEY))
print("Google key set:    ", bool(GOOGLE_API_KEY))
print("anthropic_client:  ", anthropic_client is not None)
print("google_client:     ", google_client is not None)

## 2. Load the 21 puzzles (success / partial_fail / empty_fail)

In [ ]:
import csv

CSV_PATH = "contamination_dataset.csv"   # puzzle_id, outcome, original_text, adapted_text

with open(CSV_PATH, newline="", encoding="utf-8-sig") as f:
    rows = list(csv.DictReader(f))

puzzles = []
for r in rows:
    p = {"puzzle_id": r["puzzle_id"], "outcome": r["outcome"], "original": r["original_text"].strip()}
    if r["adapted_text"].strip():
        p["adapted"] = r["adapted_text"].strip()
    puzzles.append(p)

from collections import Counter
print(f"Loaded {len(puzzles)} puzzles:", dict(Counter(p["outcome"] for p in puzzles)))
has_adapted = sum(1 for p in puzzles if "adapted" in p)
print(f"{has_adapted} of {len(puzzles)} have usable adapted text (success + partial_fail);")
print(f"{len(puzzles) - has_adapted} are original-only (empty_fail -- no parseable adaptation to compare against).")

## 3. Truncate at ~60%, split on a clause boundary, so there's a real suffix to score against

In [ ]:
def split_prefix_suffix(text: str, frac: float = 0.6) -> tuple[str, str]:
    """Cut near `frac` through the text, snapped to the nearest sentence-ish boundary
    (period, newline, or semicolon) so the model isn't asked to complete mid-word."""
    target = int(len(text) * frac)
    boundary_chars = ".\n;"
    best = target
    for i in range(target, min(target + 200, len(text))):
        if text[i] in boundary_chars:
            best = i + 1
            break
    else:
        for i in range(target, max(target - 200, 0), -1):
            if text[i] in boundary_chars:
                best = i + 1
                break
    return text[:best].strip(), text[best:].strip()

## 4. Unified completion call -- one function, dispatches by backend

In [ ]:
from tenacity import retry, wait_exponential, stop_after_attempt

@retry(wait=wait_exponential(min=2, max=30), stop=stop_after_attempt(3))
def complete(backend: str, model_id: str, reasoning: bool, system: str, user: str, max_tokens: int = 400) -> str:
    if backend == "openai":
        if reasoning:
            r = openai_client.responses.create(
                model=model_id, reasoning={"effort": "low"}, max_output_tokens=max_tokens,
                input=[{"role": "developer", "content": system}, {"role": "user", "content": user}],
            )
            return r.output_text.strip()
        r = openai_client.chat.completions.create(
            model=model_id, temperature=0.0, max_tokens=max_tokens,
            messages=[{"role": "system", "content": system}, {"role": "user", "content": user}],
        )
        return r.choices[0].message.content.strip()

    if backend == "anthropic":
        r = anthropic_client.messages.create(
            model=model_id, max_tokens=max_tokens, temperature=0.0,
            system=system, messages=[{"role": "user", "content": user}],
        )
        return r.content[0].text.strip()

    if backend == "google":
        r = google_client.models.generate_content(
            model=model_id, contents=f"{system}\n\n{user}",
            config={"temperature": 0.0, "max_output_tokens": max_tokens},
        )
        return r.text.strip()

    if backend == "ollama":
        r = ollama_client.chat.completions.create(
            model=model_id, temperature=0.0, max_tokens=max_tokens,
            messages=[{"role": "system", "content": system}, {"role": "user", "content": user}],
        )
        return r.choices[0].message.content.strip()

    raise ValueError(f"unknown backend {backend}")

## 5. Guided vs. general prompts

In [ ]:
GUIDED_SYSTEM = (
    "The following is the beginning of an Analytical Reasoning puzzle from the GRE Big Book "
    "(Educational Testing Service). Complete the passage exactly as it continues."
)
GENERAL_SYSTEM = "Complete the following passage exactly as it continues."

def build_prompt(condition: str, prefix: str) -> tuple[str, str]:
    system = GUIDED_SYSTEM if condition == "guided" else GENERAL_SYSTEM
    return system, prefix

## 6. Results cache -- load prior runs, figure out what's already done

Everything lands in one running file, `time_travel_results_master.csv`. If it doesn't exist yet but your
earlier gpt-4o/o3 CSV does, that seeds the cache so those two are already "done" on the first run.

In [ ]:
import pandas as pd

MASTER_CSV = "time_travel_results_master.csv"
SEED_CSV   = "time_travel_results_20261010_005717.csv"   # your existing gpt-4o + o3 run

RESULT_COLUMNS = ["puzzle_id", "outcome", "backend", "model", "text_type", "condition",
                   "prefix", "true_suffix", "completion"]

if os.path.exists(MASTER_CSV):
    results_df = pd.read_csv(MASTER_CSV)
    print(f"Loaded existing cache: {len(results_df)} rows -> {MASTER_CSV}")
elif os.path.exists(SEED_CSV):
    results_df = pd.read_csv(SEED_CSV)
    results_df.to_csv(MASTER_CSV, index=False)
    print(f"No cache yet -- seeded {MASTER_CSV} from {SEED_CSV} ({len(results_df)} rows)")
else:
    results_df = pd.DataFrame(columns=RESULT_COLUMNS)
    print("No cache and no seed file found -- starting from empty.")

completed_models = set(zip(results_df["backend"], results_df["model"])) if len(results_df) else set()
print("Already have results for:", sorted(completed_models) or "(none)")

In [ ]:
from tqdm import tqdm
import time as _time

def run_model_if_needed(backend: str, model_id: str, reasoning: bool) -> None:
    """Runs the full guided/general puzzle loop for one (backend, model) pair, unless it's
    already in the cache -- then it's skipped and the cache is left untouched. Saves to
    MASTER_CSV immediately after this model finishes, so a crash partway through the next
    model doesn't lose what's already been run."""
    global results_df, completed_models

    if (backend, model_id) in completed_models:
        print(f"Skipping {backend}/{model_id} -- already in {MASTER_CSV}")
        return

    print(f"Running {backend}/{model_id} ...")
    new_records = []
    for p in tqdm(puzzles, desc=f"{backend}/{model_id}"):
        text_types = ["original", "adapted"] if "adapted" in p else ["original"]
        for text_type in text_types:
            prefix, suffix = split_prefix_suffix(p[text_type])
            for condition in ["guided", "general"]:
                system, user = build_prompt(condition, prefix)
                try:
                    completion = complete(backend, model_id, reasoning, system, user)
                except Exception as e:
                    completion = f"ERROR: {e}"
                new_records.append({
                    "puzzle_id": p["puzzle_id"], "outcome": p["outcome"],
                    "backend": backend, "model": model_id,
                    "text_type": text_type, "condition": condition,
                    "prefix": prefix, "true_suffix": suffix, "completion": completion,
                })
                _time.sleep(0.3)

    new_df = pd.DataFrame(new_records)
    n_errors = new_df["completion"].str.startswith("ERROR:").sum()
    if n_errors:
        print(f"  {n_errors} rows errored for {backend}/{model_id}:")
        print("  " + new_df.loc[new_df["completion"].str.startswith("ERROR:"), "completion"].iloc[0])

    results_df = pd.concat([results_df, new_df], ignore_index=True)
    completed_models.add((backend, model_id))
    results_df.to_csv(MASTER_CSV, index=False)
    print(f"Done: {backend}/{model_id} -- {len(new_df)} rows added, cache now {len(results_df)} rows total.")

## 7. Run -- OpenAI, Anthropic, Google

Already-cached models (gpt-4o, o3 from the seed) print a skip line and cost nothing.

In [ ]:
API_MODELS = [
    ("openai",    "o3",               True),
    ("openai",    "gpt-4o",           False),
    ("anthropic", "claude-haiku-4-5", False),
    ("google",    "gemini-2.5-flash", False),
]

for backend, model_id, reasoning in API_MODELS:
    run_model_if_needed(backend, model_id, reasoning)

## 8. Ollama models -- pull, run, remove (skipped entirely if already cached)

If a model's results are already in the cache, it's never even pulled. Otherwise: pull it, run the full
loop, remove it -- one model on disk at a time. `finally` guarantees the remove step runs even if a puzzle
call fails mid-model, so a crash doesn't leave a model sitting on disk.

In [ ]:
import subprocess

OLLAMA_MODELS = ["qwen3.5:9b", "llama3.1:8b", "deepseek-r1:7b"]

def ollama_pull(model_id):
    print(f"Pulling {model_id} ...")
    subprocess.run(["ollama", "pull", model_id], check=True)

def ollama_remove(model_id):
    print(f"Removing {model_id} ...")
    subprocess.run(["ollama", "rm", model_id], check=True)

for model_id in OLLAMA_MODELS:
    if ("ollama", model_id) in completed_models:
        print(f"Skipping ollama/{model_id} -- already in {MASTER_CSV} (not pulled)")
        continue
    ollama_pull(model_id)
    try:
        run_model_if_needed("ollama", model_id, False)
    finally:
        ollama_remove(model_id)

In [ ]:
# Stop the server we started (skip if you were already running one of your own)
if ollama_proc is not None:
    ollama_proc.terminate()
    print("Ollama server stopped.")

## 9. Score: ROUGE-L overlap between completion and the true suffix

`results_df` is the full cache at this point -- everything run so far, across however many passes this notebook has had.

In [ ]:
from rouge_score import rouge_scorer
scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)

def score_row(row):
    if row["completion"].startswith("ERROR:"):
        return None
    return scorer.score(row["true_suffix"], row["completion"])["rougeL"].fmeasure

results_df["rougeL"] = results_df.apply(score_row, axis=1)

# Aggregate view: by model x text_type, collapsed across puzzles (mixes outcomes within original/adapted)
summary = (
    results_df.groupby(["backend", "model", "text_type", "condition"])["rougeL"]
    .mean().unstack("condition")
)
summary["guided_minus_general"] = summary["guided"] - summary["general"]
print(summary.round(3))
summary.to_csv(f"time_travel_summary_{RUN_ID}.csv")

# Outcome-aware view: separates success / partial_fail / empty_fail so the three-way contrast is visible
by_outcome = (
    results_df.groupby(["model", "text_type", "outcome", "condition"])["rougeL"]
    .mean().unstack("condition")
)
by_outcome["guided_minus_general"] = by_outcome["guided"] - by_outcome["general"]
print()
print(by_outcome.round(3))
by_outcome.to_csv(f"time_travel_by_outcome_{RUN_ID}.csv")

## Reading the result

For each model: compare the `guided_minus_general` gap on **original** vs. on **adapted** text.

- A clear positive gap on **original**, that shrinks toward zero on **adapted**, is the expected
  contamination signature -- the model recognizes the source puzzle specifically, and that recognition
  advantage disappears on text it's never seen.
- `partial_fail`'s one puzzle is a useful middle case: its adapted text is coherent (unlike `empty_fail`,
  which has none) but logically drifted (unlike `success`). If its guided-vs-general gap sits between the
  `success` and `empty_fail` patterns, that's a nice secondary signal.
- At n=21 (10 + 10 + 1) this is a targeted probe, not a powered study -- report raw per-puzzle scores or
  the summary table grouped by `outcome`, not a significance test.
- To force a re-run of a model you've already cached (e.g. you changed the prompt and want fresh
  completions), delete its rows from `time_travel_results_master.csv` -- or delete the whole file to start
  over from scratch -- before re-running sections 6 onward.